# Colab Training Runbook

GPU launcher for the official thesis experiments. The repository default is now the stratified curriculum setup: configs/{model}.yaml trains on data/output/train.h5 and validates on data/output/val.h5.

## Before Running

1. Select a GPU runtime: Runtime > Change runtime type > GPU.
2. Put train.h5, val.h5, and test.h5 under MyDrive/masters-thesis/data/output/.
3. Paste a GitHub token in Step 1 if the repo is private.
4. Keep GIT_REF on the branch that contains the current evaluator schema.

## Drive layout

```text
runs/<model>/<run_id>/
```

Each completed run writes best.pt, latest.pt, metrics.csv, evaluation/summary.csv, and evaluation/metrics.json.


In [ ]:
# @title 1. Setup: Drive, experiment, and repo
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")


def _shell(*cmd: str) -> None:
    """Run a command, streaming stdout/stderr live to the cell, raising on failure.

    Why Popen + manual line iteration instead of subprocess.run: the OS-level
    stdout file descriptor inherited from the kernel does not stream to the
    Colab cell in real time. Piping through Python and printing each line
    routes output via Jupyter's wrapped sys.stdout, which is the cell.
    """
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if proc.stdout is not None:
        for line in proc.stdout:
            print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


GITHUB_TOKEN = ""  # @param {type:"string"}
REPO = "AlexNeagu123/msc-thesis-gnn-nbody"
GIT_REF = "main"  # @param {type:"string"}
DRIVE = Path("/content/drive/MyDrive/masters-thesis")

RUN_MODE = "curriculum"  # @param ["curriculum"]
MODEL = "egnn"  # @param ["egnn", "hgnn"]
N_TRAIN = "1000"  # @param ["1000"]
EPOCHS = 80  # @param {type:"integer"}
NOISE_FACTORS = "0.075,0.125,0.15,0.20"  # @param {type:"string"}
RUN_TRAINING = True  # @param {type:"boolean"}
RUN_EVALUATION = True  # @param {type:"boolean"}
SKIP_COMPLETED = False  # @param {type:"boolean"}

N_TRAIN = int(N_TRAIN)
NOISE_VALUES = [float(x.strip()) for x in NOISE_FACTORS.split(",") if x.strip()]

assert RUN_MODE == "curriculum", RUN_MODE
assert MODEL in {"egnn", "hgnn"}, MODEL
assert N_TRAIN == 1000, N_TRAIN
if RUN_MODE in {"single", "noise_sweep"}:
    assert EPOCHS > 0, EPOCHS
if RUN_MODE == "noise_sweep":
    assert MODEL == "egnn", "noise_sweep is intended for EGNN only"
    assert NOISE_VALUES, "NOISE_FACTORS must contain at least one value"
print("curriculum mode: EPOCHS and NOISE_FACTORS are ignored; schedule comes from the yaml")

DATASET_NAME = "output"
DATA_PATH_PREFIX = "data/output"
if not GITHUB_TOKEN:
    raise ValueError("Paste a GitHub token before cloning the repo.")

repo_dir = Path("/content/repo")
clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if (repo_dir / ".git").exists():
    _shell("git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", GIT_REF)
    _shell("git", "-C", str(repo_dir), "checkout", "FETCH_HEAD")
elif repo_dir.exists():
    raise FileExistsError(f"{repo_dir} exists but is not a git checkout")
else:
    _shell("git", "clone", "--depth", "1", "--branch", GIT_REF, clone_url, str(repo_dir))

project_dir = repo_dir / "impl" if (repo_dir / "impl" / "requirements.txt").exists() else repo_dir
if not (project_dir / "requirements.txt").exists():
    raise FileNotFoundError(
        f"Could not find requirements.txt under {repo_dir} or {repo_dir / 'impl'}"
    )

get_ipython().run_line_magic("cd", str(project_dir))
print(f"Project directory: {project_dir}")
print(f"Run mode: {RUN_MODE}")
print(f"Model: {MODEL}")
print(f"Dataset: official stratified ({DATA_PATH_PREFIX})")
if RUN_MODE == "noise_sweep":
    print(f"Noise values: {NOISE_VALUES}")

In [ ]:
# @title 2. Install dependencies and verify GPU
_shell("pip", "install", "-q", "-r", "requirements.txt")

import torch  # noqa: E402

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime before continuing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# @title 3. Copy dataset from Drive and verify shapes
import shutil

import h5py

data_dir = Path(DATA_PATH_PREFIX)
data_dir.mkdir(parents=True, exist_ok=True)

for name in ["train.h5", "val.h5", "test.h5"]:
    src = DRIVE / "data" / "output" / name
    dst = data_dir / name
    if not src.exists():
        raise FileNotFoundError(f"Missing Drive data file: {src}")
    shutil.copy2(src, dst)

for name in ["train.h5", "val.h5", "test.h5"]:
    with h5py.File(data_dir / name, "r") as f:
        shape = f["trajectories"].shape
        print(f"{name}: {shape}")
        if name == "train.h5" and shape[0] < N_TRAIN:
            raise ValueError(f"train.h5 has {shape[0]} trajectories, but N_TRAIN={N_TRAIN}")

In [ ]:
# @title 4. Build run plan and write Colab configs
import yaml


def _source_config_path(model_name: str) -> Path:
    return Path(f"configs/{model_name}.yaml")


def _run_root(model_name: str) -> Path:
    return DRIVE / "runs" / model_name


def _write_colab_config(model_name: str, run_root: Path) -> Path:
    cfg_path = _source_config_path(model_name)
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg["data"]["train_path"] = f"{DATA_PATH_PREFIX}/train.h5"
    cfg["data"]["val_path"] = f"{DATA_PATH_PREFIX}/val.h5"
    cfg["training"]["device"] = "auto"
    cfg["checkpointing"]["enabled"] = True
    cfg["checkpointing"]["dir"] = str(run_root)
    cfg["logging"]["enabled"] = True
    cfg["logging"]["dir"] = str(run_root)

    out_path = Path("configs") / f"_colab_{model_name}_n{N_TRAIN}_curriculum.yaml"
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return out_path


cfg0 = yaml.safe_load(_source_config_path(MODEL).read_text())
RUN_SPECS = [
    {
        "model": MODEL,
        "noise_factor": float(cfg0.get("training", {}).get("noise_factor", 0.0)),
        "run_root": _run_root(MODEL),
    }
]

for spec in RUN_SPECS:
    spec["run_root"].mkdir(parents=True, exist_ok=True)
    spec["config_path"] = _write_colab_config(spec["model"], spec["run_root"])
    print(f"mode=curriculum, noise={spec['noise_factor']}: {spec['run_root']}")
    print(f"config={spec['config_path']}")


In [ ]:
# @title 5. Train and evaluate planned runs
RUN_RESULTS = []


def _train_cmd(config_path: Path) -> list[str]:
    return [
        "python",
        "-u",
        "-m",
        "training.train",
        "--config",
        str(config_path),
        "--n-train",
        str(N_TRAIN),
    ]


def _eval_cmd(config_path: Path, checkpoint: Path, eval_dir: Path) -> list[str]:
    return [
        "python",
        "-u",
        "-m",
        "evaluation.evaluate",
        "--config",
        str(config_path),
        "--checkpoint",
        str(checkpoint),
        "--test-path",
        f"{DATA_PATH_PREFIX}/test.h5",
        "--output-dir",
        str(eval_dir),
        "--device",
        "cuda",
    ]


def _train_message(model_name: str, noise_factor: float, run_root: Path) -> str:
    """Build a mode-aware status line so curriculum runs do not advertise epochs={EPOCHS}."""
    if RUN_MODE == "curriculum":
        return (
            f"training {model_name} N={N_TRAIN}, mode={RUN_MODE} "
            f"(schedule from yaml), noise={noise_factor}; output={run_root}"
        )
    return (
        f"training {model_name} N={N_TRAIN}, epochs={EPOCHS}, "
        f"noise={noise_factor}; output={run_root}"
    )


for index, spec in enumerate(RUN_SPECS, start=1):
    model_name = spec["model"]
    noise_factor = spec["noise_factor"]
    run_root = spec["run_root"]
    config_path = spec["config_path"]
    label = (
        RUN_MODE
        if RUN_MODE == "curriculum"
        else _noise_label(noise_factor)
    )

    existing = sorted(
        p for p in run_root.iterdir() if p.is_dir() and (p / "evaluation" / "metrics.json").exists()
    )
    if SKIP_COMPLETED and existing:
        run_dir = existing[-1]
        print(f"[{index}/{len(RUN_SPECS)}] skipping completed {label}: {run_dir}")
    elif RUN_TRAINING:
        print(
            f"[{index}/{len(RUN_SPECS)}] {_train_message(model_name, noise_factor, run_root)}",
            flush=True,
        )
        _shell(*_train_cmd(config_path))
        run_ids = sorted(
            p.name for p in run_root.iterdir() if p.is_dir() and (p / "latest.pt").exists()
        )
        if not run_ids:
            raise RuntimeError(f"No checkpoint runs found under {run_root}.")
        run_dir = run_root / run_ids[-1]
    else:
        run_ids = sorted(
            p.name for p in run_root.iterdir() if p.is_dir() and (p / "latest.pt").exists()
        )
        if not run_ids:
            raise RuntimeError(f"Training skipped and no checkpoint runs found under {run_root}.")
        run_dir = run_root / run_ids[-1]

    result = {
        "model": model_name,
        "noise_factor": noise_factor,
        "noise_label": label,
        "config_path": config_path,
        "run_root": run_root,
        "run_dir": run_dir,
        "best_checkpoint": run_dir / "best.pt",
        "latest_checkpoint": run_dir / "latest.pt",
        "eval_dir": run_dir / "evaluation",
    }
    RUN_RESULTS.append(result)
    print(f"ready: {label}, run_dir={run_dir}")

    if RUN_EVALUATION:
        print(
            f"[{index}/{len(RUN_SPECS)}] evaluating {label} checkpoint={result['best_checkpoint']}",
            flush=True,
        )
        _shell(*_eval_cmd(config_path, result["best_checkpoint"], result["eval_dir"]))
    else:
        print(f"Evaluation skipped for {label}.")

In [ ]:
# @title 6. Print evaluation summaries
for result in RUN_RESULTS:
    summary_path = result["eval_dir"] / "summary.csv"
    print(f"Evaluation directory: {result['eval_dir']}")
    if summary_path.exists():
        print(summary_path.read_text())

In [ ]:
# @title 7. Verify persistent Drive artifacts
for result in RUN_RESULTS:
    print(
        f"\nlabel={result['noise_label']}  noise={result['noise_factor']}  run={result['run_dir']}"
    )
    items = [
        result["best_checkpoint"],
        result["latest_checkpoint"],
        result["run_dir"] / "metrics.csv",
        result["run_dir"] / "diagnostics.log",
        result["eval_dir"] / "summary.csv",
        result["eval_dir"] / "metrics.json",
    ]

    for path in items:
        status = "ok" if path.exists() else "missing"
        print(f"{status}: {path}")

print("\nAll planned run directories:")
for result in RUN_RESULTS:
    print(result["run_dir"])